# 04. Selected genes, stability and known biology

Three checks on what the sparse models select: (1) is the selection stable across the five
training folds, (2) do the models recover biomarkers established in the literature, and (3) do
predictions from expression alone separate mutation-defined sensitive groups the model never saw?

In [1]:
import sys, warnings, pickle
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20)
plt.rcParams.update({"figure.figsize": (9, 4.2), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3})
T = ROOT / "outputs" / "tables"; FIG = ROOT / "outputs" / "figures"
from drugresponse import data

In [2]:
stab = pd.read_csv(T / "selection_stability.csv")
print("per drug, medians across drugs")
stab.groupby("model")[["median_selected", "n_all_folds", "n_3plus", "mean_jaccard"]].median().round(2)

per drug, medians across drugs


,median_selected,n_all_folds,n_3plus,mean_jaccard
model,,,,
Elastic net,50.0,6.0,29.0,0.23
Lasso,48.0,6.0,28.0,0.23


`n_all_folds`: genes selected in all five folds. `mean_jaccard`: average overlap of the selected
sets between pairs of folds (1 = identical). Each fold drops 20% of the lines, a mild perturbation,
so low overlap means the exact gene list is fragile even when accuracy is stable.

## Known biomarkers

In [3]:
bio = pd.read_csv(T / "biomarker_checks.csv")
bio.round(3)

,drug,drug_id,biomarker,kind,rationale,n,folds_gene_in_features,n_mutant,auc_mutant,auc_wildtype,roc_auc_pred_separates_mutants,rank_Elastic net,folds_Elastic net,rank_Lasso,folds_Lasso,rank_Ridge,folds_Ridge,spearman_gene_auc
0,Dabrafenib,1373,BRAF,mutation,BRAF V600 inhibitor; BRAF-mutant melanoma line...,929,NaN,80.0,0.790,0.969,0.819,NaN,0,NaN,0,NaN,0,NaN
1,Nutlin-3a (-),1047,MDM2,expression,"MDM2 inhibitor; needs wild-type TP53, whose ta...",943,0.0,NaN,NaN,NaN,NaN,NaN,0,NaN,0,NaN,0,-0.382
2,Lapatinib,1558,ERBB2,expression,EGFR/HER2 inhibitor; HER2-amplified lines are ...,941,5.0,NaN,NaN,NaN,NaN,1.0,5,1.0,5,1.0,5,-0.213
3,Venetoclax,1909,BCL2,expression,BCL-2 inhibitor; high BCL2 expression predicts...,933,5.0,NaN,NaN,NaN,NaN,1.0,5,1.0,5,1.0,5,-0.299
4,Afatinib,1032,EGFR,expression,EGFR/HER2 inhibitor; EGFR-dependent lines are ...,941,5.0,NaN,NaN,NaN,NaN,59.0,2,55.0,2,24.0,3,-0.037
5,Trametinib,1372,BRAF,mutation,MEK inhibitor; MAPK-pathway mutant lines are s...,941,NaN,82.0,0.602,0.911,0.851,NaN,0,NaN,0,NaN,0,NaN
6,Olaparib,1017,SLFN11,expression,PARP inhibitor; SLFN11 expression predicts DNA...,942,5.0,NaN,NaN,NaN,NaN,1.0,5,1.0,5,1.0,5,-0.278
7,Erlotinib,1168,EGFR,expression,EGFR inhibitor,930,5.0,NaN,NaN,NaN,NaN,1.0,5,1.0,5,1.0,5,-0.362


* `spearman_gene_auc`: marginal correlation of the biomarker gene's expression with AUC (negative =
  higher expression, more sensitive).
* `rank_*` / `folds_*`: where the biomarker gene ranks in each model's selection list, and in how
  many folds it was selected.
* For mutation biomarkers (BRAF for dabrafenib and trametinib) the models never see mutation data.
  `roc_auc_pred_separates_mutants` asks whether the elastic net's predicted AUC still ranks mutant
  lines as more sensitive (0.5 = no separation, 1 = perfect).

## Top genes for the biomarker drugs

In [4]:
top = pd.read_csv(T / "top_genes.csv", dtype={"drug": str})
drugs = data.load_drugs()
for name in bio.drug:
    d = str(bio.loc[bio.drug == name, "drug_id"].iloc[0])
    g = top[(top.drug == d) & (top.model == "Elastic net")].head(8)
    print(f"{name:<14}", ", ".join(f"{f} ({c:+.3f}, {k}/5)" for f, c, k in zip(g.feature, g.mean_coef, g.folds)))

Dabrafenib     BCL2A1 (-0.012, 5/5), S100B (-0.011, 5/5), GJB1 (-0.008, 5/5), ACP5 (-0.007, 5/5), IL16 (-0.006, 5/5), ITGB3 (-0.006, 5/5), GDF15 (-0.003, 5/5), SNTB1 (-0.002, 5/5)
Nutlin-3a (-)  CDKN1A (-0.017, 5/5), PHLDA3 (-0.015, 5/5), CDKN2A (+0.010, 5/5), TRIM22 (-0.009, 5/5), GLDC (-0.007, 5/5), ACTA2 (-0.006, 5/5), CD82 (+0.006, 5/5), MIR4435-2HG (+0.006, 5/5)
Lapatinib      ERBB2 (-0.020, 5/5), GRB7 (-0.011, 5/5), TCEAL9 (+0.006, 5/5), KRT14 (-0.004, 5/5), NR2F1 (+0.004, 5/5), CYP4F11 (-0.004, 5/5), NID1 (+0.003, 5/5), BARX1 (+0.002, 5/5)
Venetoclax     BCL2 (-0.045, 5/5), BHLHE40 (+0.010, 5/5), MILR1 (-0.007, 5/5), MYO1D (+0.006, 5/5), TRIB2 (+0.006, 5/5), IL32 (+0.004, 5/5), BEX3 (+0.004, 5/5), CHST15 (-0.004, 5/5)
Afatinib       ERBB2 (-0.025, 5/5), UNC13D (-0.007, 5/5), GRB7 (-0.007, 5/5), LDHB (-0.007, 5/5), MSLN (+0.005, 5/5), CAPN8 (-0.004, 5/5), KRT4 (-0.004, 5/5), ME3 (+0.004, 5/5)
Trametinib     LYZ (-0.026, 5/5), SPRY4 (-0.016, 5/5), ERBB2 (+0.012, 5/5), IL16 (-0.012